## Oracle AI Data Platform

Copyright © 2026, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# Build a Native Apache Iceberg Table

Create and work with an Apache Iceberg table in an existing Oracle AI Data Platform (AIDP) Standard catalog. This how-to covers writes, schema and partition evolution, an upsert with `MERGE INTO`, snapshots, time travel, and metadata inspection.

**Before you begin:** Attach this notebook to a Spark cluster and choose a Standard catalog in which you can create a schema. The notebook generates a new demo schema each time it runs, so previous runs remain untouched. Run the optional cleanup cell when you have finished exploring the table.

## 1. Choose a Standard catalog

Set `CATALOG_NAME` to an existing Standard catalog in your AIDP environment. The generated schema keeps this run separate from other examples.

In [ ]:
from uuid import uuid4

CATALOG_NAME = "default"  # Change this if your Standard catalog has another name.
SCHEMA_NAME = f"iceberg_demo_{uuid4().hex[:8]}"
TABLE_NAME = "orders"
TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

print(f"Demo table: {TABLE}")

## 2. Create a partitioned Iceberg table

`USING iceberg` selects the Iceberg table format. The `days(order_ts)` transform partitions rows by order date without adding a separate partition column to the table.

In [ ]:
spark.sql(f"CREATE SCHEMA {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id BIGINT,
        customer_id BIGINT,
        amount DECIMAL(12, 2),
        status STRING,
        order_ts TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (days(order_ts))
""")

spark.sql(f"DESCRIBE TABLE {TABLE}").show(truncate=False)

## 3. Insert and query orders

The insert commits the first data snapshot. Save its ID so that a later step can query the table as it was before the upsert.

In [ ]:
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        (1, 10, 120.00, 'NEW', TIMESTAMP '2026-09-01 10:00:00'),
        (2, 11, 80.50, 'NEW', TIMESTAMP '2026-09-02 11:30:00')
""")

INITIAL_SNAPSHOT_ID = spark.sql(f"""
    SELECT snapshot_id
    FROM {TABLE}.snapshots
    ORDER BY committed_at ASC
    LIMIT 1
""").first().snapshot_id

spark.sql(f"SELECT order_id, customer_id, amount, status FROM {TABLE} ORDER BY order_id").show()

## 4. Evolve the schema and partition specification

Add a nullable `channel` column. Existing rows read as `NULL` in that column. Then add a bucket transform for `customer_id`; subsequent writes use the updated partition specification.

In [ ]:
spark.sql(f"ALTER TABLE {TABLE} ADD COLUMN channel STRING")
spark.sql(f"ALTER TABLE {TABLE} ADD PARTITION FIELD bucket(16, customer_id)")

spark.sql(f"SELECT order_id, channel FROM {TABLE} ORDER BY order_id").show()

## 5. Upsert with `MERGE INTO`

Match on `order_id`: update the amount and status of order `2`, and insert order `3`. The source rows also populate the new `channel` column.

In [ ]:
spark.sql(f"""
    MERGE INTO {TABLE} AS target
    USING (
        SELECT * FROM VALUES
            (2, 11, CAST(95.00 AS DECIMAL(12, 2)), 'PAID',
             TIMESTAMP '2026-09-02 11:30:00', 'web'),
            (3, 12, CAST(40.00 AS DECIMAL(12, 2)), 'NEW',
             TIMESTAMP '2026-09-03 12:00:00', 'store')
        AS source(order_id, customer_id, amount, status, order_ts, channel)
    ) AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET
        target.amount = source.amount,
        target.status = source.status,
        target.channel = source.channel
    WHEN NOT MATCHED THEN INSERT
        (order_id, customer_id, amount, status, order_ts, channel)
        VALUES (source.order_id, source.customer_id, source.amount,
                source.status, source.order_ts, source.channel)
""")

orders = spark.sql(f"""
    SELECT order_id, amount, status, channel
    FROM {TABLE}
    ORDER BY order_id
""")
orders.show()

actual = [(row.order_id, str(row.amount), row.status, row.channel)
          for row in orders.collect()]
assert actual == [
    (1, '120.00', 'NEW', None),
    (2, '95.00', 'PAID', 'web'),
    (3, '40.00', 'NEW', 'store'),
], actual

## 6. Inspect snapshots and query an earlier version

Each data write creates a snapshot. Compare the current table with the snapshot saved immediately after the first insert.

In [ ]:
spark.sql(f"""
    SELECT snapshot_id, committed_at, operation
    FROM {TABLE}.snapshots
    ORDER BY committed_at
""").show(truncate=False)

before_merge = spark.sql(f"""
    SELECT order_id, amount, status
    FROM {TABLE} VERSION AS OF {INITIAL_SNAPSHOT_ID}
    ORDER BY order_id
""")
before_merge.show()
assert before_merge.count() == 2
assert spark.sql(f"SELECT COUNT(*) AS row_count FROM {TABLE}").first().row_count == 3

## 7. Inspect the current data files

Iceberg metadata tables expose information about the files in the current snapshot. Summarize their format and row counts without opening the files directly.

In [ ]:
spark.sql(f"""
    SELECT file_format, COUNT(*) AS data_files,
           SUM(record_count) AS records,
           SUM(file_size_in_bytes) AS bytes
    FROM {TABLE}.files
    GROUP BY file_format
""").show()

## Optional: Remove the demo schema

Run the next cell after you finish exploring the table. It removes the unique schema generated by this notebook and its demo table.

In [ ]:
spark.sql(f"DROP SCHEMA {CATALOG_NAME}.{SCHEMA_NAME} CASCADE")
print("Demo schema removed.")

## What you learned

You created a native Iceberg table in a Standard catalog, inserted and upserted orders, evolved its schema and partition specification, compared snapshots with time travel, and inspected file metadata. To adapt this workflow, replace the sample orders with your own data and choose partition transforms that match your query patterns.